# LLMOps / GenAI Production> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. Környezet

In [ ]:
%pip install mlflow openai --quiet

## 2. Prompt versioning MLflow-val

In [ ]:
import mlflow, timemlflow.set_experiment('webshop/chatbot-prompts')prompts = {    'v1': 'Segítőkész asszisztens vagy. Válaszolj röviden.',    'v2': 'Segítőkész magyar asszisztens vagy egy webshopnak. Udvariasan és pontosan válaszolj, a kontextus alapján.',    'v3': 'Magyar ügyfélszolgálat vagy. Csak a megadott kontextusból válaszolj. Ha nincs benne a válasz, mondd: "Sajnálom, erre nem tudok válaszolni."',}for version, system_prompt in prompts.items():    with mlflow.start_run(run_name=f'prompt_{version}'):        mlflow.log_param('version', version)        mlflow.log_param('prompt_length', len(system_prompt))        mlflow.log_text(system_prompt, f'prompts/{version}.txt')        # Szimulált eval        metrics = {            'v1': {'accuracy': 0.72, 'avg_latency_ms': 450, 'tokens': 120},            'v2': {'accuracy': 0.85, 'avg_latency_ms': 520, 'tokens': 180},            'v3': {'accuracy': 0.92, 'avg_latency_ms': 510, 'tokens': 220},        }[version]        mlflow.log_metrics(metrics)        print(f'  {version}: accuracy={metrics["accuracy"]}, latency={metrics["avg_latency_ms"]}ms')print('\nNézd meg az eredményeket:')print('  mlflow ui --backend-store-uri ./mlruns')

## 3. Eval dataset automatizált futtatása

In [ ]:
eval_dataset = [    {'q': 'Mennyi a visszaküldési határidő?',  'expected_keyword': '14 nap'},    {'q': 'Mennyibe kerül a szállítás?',       'expected_keyword': '1490'},    {'q': 'Lehet-e bankkártyával fizetni?',    'expected_keyword': 'bankkárty'},    {'q': 'Hány hónap a garancia?',            'expected_keyword': '24'},]def simulate_response(q, prompt_version):    # Szimuláljuk a válasz minőségét a prompt verziójától függően    import random    random.seed(hash(q + prompt_version))    quality = {'v1': 0.72, 'v2': 0.85, 'v3': 0.92}[prompt_version]    return random.random() < qualitydef run_eval(prompt_version):    scores = [simulate_response(item['q'], prompt_version) for item in eval_dataset]    return sum(scores) / len(scores)print('Prompt verzió | Pass rate')print('─' * 30)for v in ['v1', 'v2', 'v3']:    print(f'   {v}         |    {run_eval(v):.0%}')

## 4. Cost & latency tracking

In [ ]:
import time, randomfrom statistics import mean, quantilesdef log_llm_call(model, tokens_in, tokens_out, latency_ms):    # Token árazás (példa, 2025-ös OpenAI árak)    prices = {        'gpt-4o':      {'in': 0.0025 / 1000, 'out': 0.01   / 1000},        'gpt-4o-mini': {'in': 0.00015 / 1000, 'out': 0.0006 / 1000},    }    p = prices[model]    cost_usd = tokens_in * p['in'] + tokens_out * p['out']    return {'model': model, 'tokens_in': tokens_in, 'tokens_out': tokens_out,            'latency_ms': latency_ms, 'cost_usd': cost_usd}# Szimulált 100 híváscalls = []for _ in range(100):    calls.append(log_llm_call(        'gpt-4o-mini',        random.randint(200, 500),        random.randint(50, 200),        random.randint(300, 1500),    ))latencies = [c['latency_ms'] for c in calls]p50, p95 = quantiles(latencies, n=100)[49], quantiles(latencies, n=100)[94]print(f'Hívások:         {len(calls)}')print(f'Átlag latency:   {mean(latencies):.0f} ms')print(f'p50 latency:     {p50:.0f} ms')print(f'p95 latency:     {p95:.0f} ms')print(f'Total cost:      ${sum(c["cost_usd"] for c in calls):.4f}')print(f'Avg cost / call: ${mean(c["cost_usd"] for c in calls):.5f}')

## 5. Fallback model pattern

In [ ]:
class LLMClient:    def __init__(self, primary='gpt-4o-mini', fallback='gpt-4o'):        self.primary = primary        self.fallback = fallback    def chat(self, messages, max_retries=1):        # Egyszerűsített — valódi kódban timeout, rate-limit, stb.        try:            return self._call(self.primary, messages)        except Exception as e:            print(f'  primary failed: {e}; fallback {self.fallback}')            return self._call(self.fallback, messages)    def _call(self, model, messages):        # Szimulált failure        if model == 'gpt-4o-mini' and random.random() < 0.3:            raise RuntimeError('rate limit')        return {'model': model, 'response': 'szimulált válasz', 'cost': 0.001}client = LLMClient()for _ in range(5):    r = client.chat([{'role': 'user', 'content': 'ping'}])    print(r)

## Következő lépések- Térj vissza a [web-alapú kurzushoz](llmops-genai-production/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*